In [1]:
import pandas as pd

file_path = "/Users/huahua/Desktop/train.csv"

# Read the CSV file
df = pd.read_csv(file_path, delimiter=";")

# Display first few rows
print(df.head())


   age           job  marital  education default  balance housing loan  \
0   58    management  married   tertiary      no     2143     yes   no   
1   44    technician   single  secondary      no       29     yes   no   
2   33  entrepreneur  married  secondary      no        2     yes  yes   
3   47   blue-collar  married    unknown      no     1506     yes   no   
4   33       unknown   single    unknown      no        1      no   no   

   contact  day month  duration  campaign  pdays  previous poutcome   y  
0  unknown    5   may       261         1     -1         0  unknown  no  
1  unknown    5   may       151         1     -1         0  unknown  no  
2  unknown    5   may        76         1     -1         0  unknown  no  
3  unknown    5   may        92         1     -1         0  unknown  no  
4  unknown    5   may       198         1     -1         0  unknown  no  


/var/folders/3j/51w1rjp53fd_5rt5m6xczh4w0000gn/T/ipykernel_93930/2330785831.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


# generate conversion rate column

In [2]:
# Convert 'y' to binary (1 for 'yes', 0 for 'no')
df['conversion_binary'] = df['y'].apply(lambda x: 1 if x == 'yes' else 0)

# Calculate conversion rate (success rate per contact attempt)
df['conversion_rate'] = df['conversion_binary'] / df['campaign']

# Display the first few rows
print(df[['y', 'campaign', 'conversion_rate']].head())

    y  campaign  conversion_rate
0  no         1              0.0
1  no         1              0.0
2  no         1              0.0
3  no         1              0.0
4  no         1              0.0


In [3]:
# Sort dataframe by conversion_rate in descending order
df_sorted = df.sort_values(by='conversion_rate', ascending=False)

# Display the first few rows after sorting
print(df_sorted[['y', 'campaign', 'conversion_rate']].head())


         y  campaign  conversion_rate
34050  yes         1              1.0
8731   yes         1              1.0
36792  yes         1              1.0
42835  yes         1              1.0
42834  yes         1              1.0


# generate best time to contact col

In [4]:
contact_time_mapping = {
    "student": "6-8pm",
    "retired": "12-2pm",
    "unemployed": "12-2pm",
    "housemaid": "2-4pm",
    "admin.": "4-5pm",
    "management": "4-5pm",
    "entrepreneur": "4-5pm",
    "blue-collar": "4-5pm",
    "self-employed": "4-5pm",
    "technician": "4-5pm",
    "services": "4-5pm",
    "unknown": "4-5pm"
}

# Apply mapping to create the new column
df["best_contact_time"] = df["job"].map(contact_time_mapping)

# Display result
print(df[["job", "best_contact_time"]].head())

            job best_contact_time
0    management             4-5pm
1    technician             4-5pm
2  entrepreneur             4-5pm
3   blue-collar             4-5pm
4       unknown             4-5pm


# generate fatigue score

In [5]:
# Define decay factor (adjust as needed)
df["decay_factor"] = df.apply(lambda row: 0.7 if (row["campaign"] + row["previous"]) > 5 else 0.5, axis=1)

# Calculate Fatigue Score
df["fatigue_score"] = (df["campaign"] + df["previous"]) * df["decay_factor"]

# Display the results
print(df[["campaign", "previous", "decay_factor", "fatigue_score"]].head())


   campaign  previous  decay_factor  fatigue_score
0         1         0           0.5            0.5
1         1         0           0.5            0.5
2         1         0           0.5            0.5
3         1         0           0.5            0.5
4         1         0           0.5            0.5


In [6]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from scipy.stats import beta

# -----------------------------
# STEP 1: LOAD & PREPROCESS DATA
# -----------------------------

# Encode categorical variables
label_encoders = {}
categorical_cols = ["job", "marital", "education", "default", "housing", "loan", "contact", "month", "poutcome", "y"]

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Feature scaling for numeric columns
scaler = StandardScaler()
df[['age', 'balance', 'duration', 'campaign', 'pdays', 'previous', 'conversion_rate', 'fatigue_score']] = \
    scaler.fit_transform(df[['age', 'balance', 'duration', 'campaign', 'pdays', 'previous', 'conversion_rate', 'fatigue_score']])

# -----------------------------
# STEP 2: FEATURE ENGINEERING
# -----------------------------
# Time-based segmentation
df['is_weekend'] = df['day'].apply(lambda x: 1 if x in [6, 7] else 0)
df['time_of_day'] = pd.cut(df['duration'], bins=[0, 180, 600, 1800, np.inf], labels=['short', 'medium', 'long', 'very_long'])

# Fatigue-adjusted engagement score
df['adjusted_engagement'] = df['conversion_rate'] * (1 - df['fatigue_score'])

# Customer clustering (Basic Segmentation)
df['customer_segment'] = pd.cut(df['balance'], bins=[-np.inf, 0, 5000, 20000, np.inf], labels=['low', 'mid', 'high', 'very_high'])

# -----------------------------
# STEP 3: DEFINE CAMPAIGN ACTIONS
# -----------------------------
campaign_actions = [
    {'offer': 'discount', 'contact_method': 'telephone', 'time': 'short'},
    {'offer': 'bonus', 'contact_method': 'cellular', 'time': 'medium'},
    {'offer': 'no_offer', 'contact_method': 'telephone', 'time': 'long'},
    {'offer': 'free_trial', 'contact_method': 'cellular', 'time': 'very_long'}
]

# Initialize Bayesian reward tracking
bandit_rewards = {i: {'successes': 1, 'failures': 1} for i in range(len(campaign_actions))}

# -----------------------------
# STEP 4: THOMPSON SAMPLING MAB
# -----------------------------
def select_best_action():
    """ Selects the best campaign action using Bayesian Thompson Sampling """
    sampled_values = [beta.rvs(bandit_rewards[i]['successes'], bandit_rewards[i]['failures']) for i in range(len(campaign_actions))]
    return np.argmax(sampled_values)

# Assign strategy dynamically
df['best_strategy'] = df.apply(lambda row: select_best_action(), axis=1)

# Map selected action details
df['best_offer'] = df['best_strategy'].apply(lambda x: campaign_actions[x]['offer'])
df['best_contact_method'] = df['best_strategy'].apply(lambda x: campaign_actions[x]['contact_method'])
df['best_contact_time'] = df['best_strategy'].apply(lambda x: campaign_actions[x]['time'])

# -----------------------------
# STEP 5: REWARD FUNCTION (MULTI-OBJECTIVE)
# -----------------------------
df['conversion_reward'] = df['y']
df['fatigue_penalty'] = df['fatigue_score'] * -0.2  # Penalize high fatigue scores
df['engagement_reward'] = df['adjusted_engagement']

# Final weighted reward
df['final_reward'] = 0.6 * df['conversion_reward'] + 0.3 * df['engagement_reward'] + 0.1 * df['fatigue_penalty']

# Update Bandit Rewards
for idx, row in df.iterrows():
    action_idx = row['best_strategy']
    if row['final_reward'] > 0.5:
        bandit_rewards[action_idx]['successes'] += 1
    else:
        bandit_rewards[action_idx]['failures'] += 1

# -----------------------------
# STEP 6: DRIFT DETECTION & ADAPTATION
# -----------------------------
def detect_drift(df, threshold=0.1):
    """ Detects drift in campaign engagement and conversion rates """
    recent_conversion_rates = df.groupby('best_strategy')['conversion_reward'].mean()
    if recent_conversion_rates.std() > threshold:
        print("🚨 Drift detected! Adjusting strategies...")
        return True
    return False

if detect_drift(df):
    print("Re-training needed!")

# -----------------------------
# STEP 7: OUTPUT RECOMMENDATIONS
# -----------------------------
df[['age', 'job', 'marital', 'education', 'best_offer', 'best_contact_method', 'best_contact_time']].to_csv("optimized_campaign_strategies.csv", index=False)

print("✅ Strategy Optimization Completed!")


✅ Strategy Optimization Completed!
